In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import scipy
from scipy.sparse import csr_matrix
import yaml

import anndata as an
import scanpy as sc

sc.settings.verbosity = 2

In [2]:
# dpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/hsc_epi2me/"
dpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/hsc_epi2me_full/"
subdirs = glob.glob(f"{dpath}/*_*")

data = {}

for path in subdirs:
    group_id = path.split("/")[-1]

    # if not 'hac' in group_id:
    #     continue
    
    mtx_path = os.path.join(path, f"{group_id}.gene_raw_feature_bc_matrix")
    df = sc.read_10x_mtx(mtx_path).to_df()
    print(f"{group_id} {df.shape=}")
    data[group_id] = df

data.keys()

hac_v3 df.shape=(8576, 22330)
hac_v4.3 df.shape=(8587, 18742)
hac_v4.2 df.shape=(8657, 23924)


dict_keys(['hac_v3', 'hac_v4.3', 'hac_v4.2'])

# COMBINE

In [3]:
# Step 1: Get union of all indices and columns
all_rows = set()
all_cols = set()
for df in data.values():
    all_rows.update(df.index)
    all_cols.update(df.columns)

# Convert to sorted lists for consistency
all_rows = sorted(all_rows)
all_cols = sorted(all_cols)

# Step 2: Create a zero-filled DataFrame with full index and columns
X = pd.DataFrame(0, index=all_rows, columns=all_cols, dtype=np.float64)
print(f"{X.shape=}")

# Step 3: Add values from each DataFrame
obs = []
var = []

for group_id, df in data.items():
    aligned = df.reindex(
        index=all_rows, 
        columns=all_cols,
        fill_value=0)
    
    X += aligned

    # add the total number of reads from each basecall model
    obs_sum = pd.DataFrame(aligned.sum(axis=1), columns=[group_id])
    var_sum = pd.DataFrame(aligned.sum(axis=0), columns=[group_id])

    obs.append(obs_sum)
    var.append(var_sum)
    
# compile
X = csr_matrix(X.to_numpy())
print(f"{X.shape=}")

obs = pd.concat(obs, ignore_index=False, axis=1)
var = pd.concat(var, ignore_index=False, axis=1)

print(f"{obs.shape=}")
print(f"{var.shape=}")

# build anndata
adata = an.AnnData(
    X=X, 
    obs=obs,
    var=var,
)

adata

X.shape=(8693, 25149)
X.shape=(8693, 25149)
obs.shape=(8693, 3)
var.shape=(25149, 3)


AnnData object with n_obs × n_vars = 8693 × 25149
    obs: 'hac_v3', 'hac_v4.3', 'hac_v4.2'
    var: 'hac_v3', 'hac_v4.3', 'hac_v4.2'

# Write to File

In [4]:
# outpath = "/nfs/turbo/umms-indikar/shared/projects/HSC/pipeline_outputs/hsc_epi2me_full/anndata/gene_adata_hac_only.h5ad"
# adata.write(outpath)
# adata

AnnData object with n_obs × n_vars = 8693 × 25149
    obs: 'hac_v3', 'hac_v4.3', 'hac_v4.2'
    var: 'hac_v3', 'hac_v4.3', 'hac_v4.2'